In [ ]:
import pandas as pd
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import numpy as np

In [ ]:
# 1. Load the data
df = pd.read_csv('/home/edu/Dropbox/Edu/repositorios/salario-minimo-y-pobreza-Ecuador/indices_region.csv', encoding='latin-1')

In [ ]:
# 2. Clean and Filter
# Exclude Galapagos (NaNs) and the chaotic year 2000
df = df[df['region'] != 'Galápagos']
df = df[df['ano'] >= 2001] 

In [ ]:
# Create a 'Date' index for the Panel
df['date'] = pd.to_datetime(df['ano'].astype(str) + 'Q' + df['trimestre'].astype(str))
df = df.set_index(['region', 'date'])

/tmp/ipykernel_71855/3453128801.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['ano'].astype(str) + 'Q' + df['trimestre'].astype(str))


# Naive models

In [ ]:
# 3. Define the Model (Two-Way Fixed Effects)
# EntityEffects = Region Fixed Effects
# TimeEffects = Time Fixed Effects
mod = PanelOLS.from_formula('fgt0 ~ kaitz_indice + EntityEffects + TimeEffects', data=df)

# 4. Run the Regression
# clustered_entity=True handles the correlation of errors within regions (Standard Practice)
res = mod.fit(cov_type='clustered', cluster_entity=True)

print(res)

                          PanelOLS Estimation Summary                           
Dep. Variable:                   fgt0   R-squared:                        0.3316
Estimator:                   PanelOLS   R-squared (Between):              0.7159
No. Observations:                 877   R-squared (Within):              -0.0473
Date:                Mon, Feb 23 2026   R-squared (Overall):              0.6473
Time:                        15:10:25   Log-likelihood                    1479.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      382.07
Entities:                          13   P-value                           0.0000
Avg Obs:                       67.462   Distribution:                   F(1,770)
Min Obs:                       4.0000                                           
Max Obs:                       94.000   F-statistic (robust):             40.961
                            

## Robustness

In [ ]:
# Create necessary variables for the checks
df['kaitz_sq'] = df['kaitz_indice'] ** 2
df['log_salario'] = np.log(df['salario_minimo'])
df['log_mediana'] = np.log(df['ingreso_mediana'])
df['const'] = 1  # Constant term is often required/good practice

In [ ]:
# ==============================================================================
# CHECK 1: The Inequality Trade-off (Dependent Variable: a50)
# ==============================================================================
print("\n" + "="*80)
print("CHECK 1: INEQUALITY TRADE-OFF (Dependent Variable: a50)")
print("Hypothesis: Does Min Wage reduce inequality even if it increases poverty?")
print("="*80)

mod_ineq = PanelOLS(df['a50'], df[['const', 'kaitz_indice']], entity_effects=True, time_effects=True)
res_ineq = mod_ineq.fit(cov_type='clustered', cluster_entity=True)
print(res_ineq)


CHECK 1: INEQUALITY TRADE-OFF (Dependent Variable: a50)
Hypothesis: Does Min Wage reduce inequality even if it increases poverty?
                          PanelOLS Estimation Summary                           
Dep. Variable:                    a50   R-squared:                        0.0481
Estimator:                   PanelOLS   R-squared (Between):              0.1351
No. Observations:                 877   R-squared (Within):              -0.0230
Date:                Mon, Feb 23 2026   R-squared (Overall):             -0.0163
Time:                        15:19:58   Log-likelihood                    1805.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      38.892
Entities:                          13   P-value                           0.0000
Avg Obs:                       67.462   Distribution:                   F(1,770)
Min Obs:                       4.0000                      

/home/edu/Dropbox/Edu/repositorios/salario-minimo-y-pobreza-Ecuador/.venv/lib/python3.12/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [ ]:
# ==============================================================================
# CHECK 2: Non-Linearity (The "U" Shape)
# ==============================================================================
print("\n" + "="*80)
print("CHECK 2: NON-LINEARITY (Testing for 'Optimal' Level)")
print("Hypothesis: Is the relationship U-shaped? (Good at low levels, bad at high?)")
print("="*80)

mod_sq = PanelOLS(df['fgt0'], df[['const', 'kaitz_indice', 'kaitz_sq']], entity_effects=True, time_effects=True)
res_sq = mod_sq.fit(cov_type='clustered', cluster_entity=True)
print(res_sq)


CHECK 2: NON-LINEARITY (Testing for 'Optimal' Level)
Hypothesis: Is the relationship U-shaped? (Good at low levels, bad at high?)
                          PanelOLS Estimation Summary                           
Dep. Variable:                   fgt0   R-squared:                        0.5452
Estimator:                   PanelOLS   R-squared (Between):              0.6385
No. Observations:                 877   R-squared (Within):              -0.2951
Date:                Mon, Feb 23 2026   R-squared (Overall):              0.0483
Time:                        15:21:03   Log-likelihood                    1648.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      460.99
Entities:                          13   P-value                           0.0000
Avg Obs:                       67.462   Distribution:                   F(2,769)
Min Obs:                       4.0000                      

/home/edu/Dropbox/Edu/repositorios/salario-minimo-y-pobreza-Ecuador/.venv/lib/python3.12/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [ ]:
# ==============================================================================
# CHECK 3 (CORRECTED): The Denominator Effect
# ==============================================================================

# PATH A: The Rigorous "Denominator" Test
# We keep TimeEffects to control for national shocks, but we only look at local income.
print("\n" + "="*80)
print("CHECK 3A: The Denominator Test (Local Income Shocks)")
print("Hypothesis: Does a drop in local median income significantly increase poverty?")
print("If YES -> The 'Perverse' Kaitz result is likely due to income shocks (Denominator Effect).")
print("="*80)

# Note: We removed 'log_salario' and 'const' to avoid absorption
mod_denom = PanelOLS(df['fgt0'], df[['log_mediana']], entity_effects=True, time_effects=True)
res_denom = mod_denom.fit(cov_type='clustered', cluster_entity=True)
print(res_denom)


# PATH B: The "Naive" Decomposition
# We DROP TimeEffects so we can see the Minimum Wage coefficient.
print("\n" + "="*80)
print("CHECK 3B: Naive Decomposition (No Time Fixed Effects)")
print("Hypothesis: What is the raw correlation of Min Wage vs Poverty?")
print("Warning: This does not control for other national trends (like oil prices).")
print("="*80)

# We include 'const' here because we don't have TimeEffects to act as intercepts
mod_naive = PanelOLS(df['fgt0'], df[['const', 'log_salario', 'log_mediana']], entity_effects=True, time_effects=False)
res_naive = mod_naive.fit(cov_type='clustered', cluster_entity=True)
print(res_naive)


CHECK 3A: The Denominator Test (Local Income Shocks)
Hypothesis: Does a drop in local median income significantly increase poverty?
If YES -> The 'Perverse' Kaitz result is likely due to income shocks (Denominator Effect).
                          PanelOLS Estimation Summary                           
Dep. Variable:                   fgt0   R-squared:                        0.5309
Estimator:                   PanelOLS   R-squared (Between):             -18.696
No. Observations:                 877   R-squared (Within):               0.8554
Date:                Mon, Feb 23 2026   R-squared (Overall):             -17.161
Time:                        15:23:13   Log-likelihood                    1634.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      871.32
Entities:                          13   P-value                           0.0000
Avg Obs:                       67.462   Distrib

/home/edu/Dropbox/Edu/repositorios/salario-minimo-y-pobreza-Ecuador/.venv/lib/python3.12/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


## Lagged model

In [ ]:
# ==============================================================================
# FINAL CHECK: THE LAGGED MODEL (Dynamic Effects)
# ==============================================================================

# 1. Create Lagged Variables
# We group by region to ensure we don't shift data from one province to another
df['kaitz_lag1'] = df.groupby('region')['kaitz_indice'].shift(1) # 3 Months
df['kaitz_lag2'] = df.groupby('region')['kaitz_indice'].shift(2) # 6 Months
df['kaitz_lag4'] = df.groupby('region')['kaitz_indice'].shift(4) # 1 Year

df['log_salario_lag2'] = df.groupby('region')['log_salario'].shift(2) # Wage 6 months ago
df['log_mediana_lag2'] = df.groupby('region')['log_mediana'].shift(2) # Income 6 months ago

# Drop NaN values created by shifting (we lose the first year of data)
df_lagged = df.dropna(subset=['kaitz_lag4', 'log_salario_lag2']).copy()

print("\n" + "="*80)
print("MODEL 1: Distributed Lag Model (Kaitz Index)")
print("Hypothesis: Does the 'bite' reduce poverty after a delay (3, 6, or 12 months)?")
print("Look for NEGATIVE coefficients on the lags.")
print("="*80)

# We include Current + Lags to see the full timeline
mod_dynamic = PanelOLS(df_lagged['fgt0'], 
                       df_lagged[['const', 'kaitz_indice', 'kaitz_lag1', 'kaitz_lag2', 'kaitz_lag4']], 
                       entity_effects=True, time_effects=True)
res_dynamic = mod_dynamic.fit(cov_type='clustered', cluster_entity=True)
print(res_dynamic)


print("\n" + "="*80)
print("MODEL 2: The 6-Month Wage Test (Decomposed)")
print("Hypothesis: Does a raw Minimum Wage hike today reduce poverty 6 months later?")
print("Look at 'log_salario_lag2'.")
print("="*80)

# No TimeEffects here to avoid absorption of the national wage
mod_wage_lag = PanelOLS(df_lagged['fgt0'], 
                        df_lagged[['const', 'log_salario_lag2', 'log_mediana_lag2']], 
                        entity_effects=True, time_effects=False)
res_wage_lag = mod_wage_lag.fit(cov_type='clustered', cluster_entity=True)
print(res_wage_lag)


MODEL 1: Distributed Lag Model (Kaitz Index)
Hypothesis: Does the 'bite' reduce poverty after a delay (3, 6, or 12 months)?
Look for NEGATIVE coefficients on the lags.
                          PanelOLS Estimation Summary                           
Dep. Variable:                   fgt0   R-squared:                        0.4112
Estimator:                   PanelOLS   R-squared (Between):              0.7466
No. Observations:                 777   R-squared (Within):              -0.8057
Date:                Mon, Feb 23 2026   R-squared (Overall):             -0.0805
Time:                        15:30:49   Log-likelihood                    1394.3
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      118.37
Entities:                          10   P-value                           0.0000
Avg Obs:                       77.700   Distribution:                   F(4,678)
Min Obs:             

/home/edu/Dropbox/Edu/repositorios/salario-minimo-y-pobreza-Ecuador/.venv/lib/python3.12/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
/home/edu/Dropbox/Edu/repositorios/salario-minimo-y-pobreza-Ecuador/.venv/lib/python3.12/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Testing the trend

In [ ]:
# ==============================================================================
# THE FINAL STRESS TEST: Controlling for the Time Trend
# ==============================================================================

# 1. Create a Linear Time Trend variable
# We assign a number to each date (0, 1, 2, 3...)
dates = df_lagged.index.get_level_values('date').unique().sort_values()
date_map = {date: i for i, date in enumerate(dates)}
df_lagged['time_trend'] = df_lagged.index.get_level_values('date').map(date_map)

print("\n" + "="*80)
print("STRESS TEST: Model 2 with Linear Time Trend")
print("Hypothesis: Does the wage effect survive when we control for the general passage of time?")
print("Look at 'log_salario_lag2'.")
print("="*80)

# We add 'time_trend' to the model
mod_trend = PanelOLS(df_lagged['fgt0'], 
                     df_lagged[['const', 'log_salario_lag2', 'log_mediana_lag2', 'time_trend']], 
                     entity_effects=True, time_effects=False)
res_trend = mod_trend.fit(cov_type='clustered', cluster_entity=True)
print(res_trend)


STRESS TEST: Model 2 with Linear Time Trend
Hypothesis: Does the wage effect survive when we control for the general passage of time?
Look at 'log_salario_lag2'.
                          PanelOLS Estimation Summary                           
Dep. Variable:                   fgt0   R-squared:                        0.5529
Estimator:                   PanelOLS   R-squared (Between):              0.6482
No. Observations:                 780   R-squared (Within):               0.5529
Date:                Mon, Feb 23 2026   R-squared (Overall):              0.5399
Time:                        15:35:15   Log-likelihood                    983.14
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      316.21
Entities:                          10   P-value                           0.0000
Avg Obs:                       78.000   Distribution:                   F(3,767)
Min Obs:                   

/home/edu/Dropbox/Edu/repositorios/salario-minimo-y-pobreza-Ecuador/.venv/lib/python3.12/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


# Sotomayor